# DesignJanus: Pilot Analysis

After running the pilot with 5–8 raters, use this notebook to:
- Check inter-rater agreement
- Identify redundant categories
- Spot confusing prompts
- Estimate annotation time

In [ ]:
import pandas as pd
from pathlib import Path

REPO = Path("..")
ANNOTATIONS = REPO / "data" / "annotations"

# Load all annotation CSVs (exclude merged)
dfs = []
for p in ANNOTATIONS.glob("*.csv"):
    if p.name == "merged_results.csv":
        continue
    df = pd.read_csv(p)
    dfs.append(df)

if dfs:
    pilot = pd.concat(dfs, ignore_index=True)
    print(pilot.shape)
    pilot.head()
else:
    print("No annotation files found.")

## Inter-rater agreement (samples rated by multiple people)

In [ ]:
# For samples with 2+ raters, compute correlation or Krippendorff's alpha
dims = ["overall", "identity", "geometry", "parts", "material", "silhouette", "usefulness"]

if "pilot" in dir() and len(pilot) > 0:
    overlap = pilot.groupby(["method", "prompt_id"]).size()
    overlap = overlap[overlap >= 2]
    print(f"Samples with 2+ raters: {len(overlap)}")
    
    # Simple: mean absolute difference between raters
    for dim in dims:
        if dim in pilot.columns:
            agg = pilot.groupby(["method", "prompt_id"])[dim].agg(["mean", "std", "count"])
            agg = agg[agg["count"] >= 2]
            if len(agg) > 0:
                print(f"{dim}: mean std = {agg['std'].mean():.2f}")

## Category correlations (redundancy check)

In [ ]:
if "pilot" in dir() and len(pilot) > 0 and all(d in pilot.columns for d in dims):
    corr = pilot[dims].corr()
    display(corr)
    # High correlation (>0.9) suggests merging

## Failure tag frequency

In [ ]:
if "pilot" in dir() and "failure_tags" in pilot.columns:
    tags = pilot["failure_tags"].str.split("|").explode().dropna()
    tags = tags[tags != ""]
    print(tags.value_counts())